# StressNet pretrained model finetuning

Note: setup tensorflow with GPU to get faster results (this example was run on a laptop without GPU).

For bigger datasets, consider using higher batch sizes with the model loaded in disjoint mode (examples coming soon).

### Import necessary packages

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import stressnet
from stressnet import data_utils
from os import makedirs
from os.path import isdir
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, TerminateOnNaN
from tensorflow.keras.metrics import MeanAbsoluteError, MeanAbsolutePercentageError
from tensorflow.keras.losses import MeanSquaredError
from matplotlib import pyplot as plt

### Download the example finetuning dataset the first time

This dataset includes 300 simulations for training and validation and 100 for testing, all from the pattern family `Radial 1D/2D Inverted`. This pattern family was not included in the training set of the pretrained StressNET models.

In [2]:
from download_radial_1d_2d_inverted_9v import ensure_dataset_downloaded
ensure_dataset_downloaded()

Dataset already available. Use force=True (or --force flag) to delete and download again.


### Configure data sources, base weights and training hyperparameters

In [3]:
SEED = 1337
np.random.seed(SEED)
tf.random.set_seed(SEED)

Note: training for a fixed number of epochs (e.g. 20) is recommended instead of using early-stopping for smaller datasets.

In [4]:
BASE_MODEL_WEIGHTS = 'STRESSNET-PRETRAINED-A'

SUBSAMPLE = 1.0  # use 100% of the training data
VAL_SPLIT = 0.25  # use 25% of the data for validation / early-stopping 
LOSS = MeanSquaredError()
METRICS = [MeanAbsoluteError(), MeanAbsolutePercentageError()]

# freeze all layers except for these 3
NET_KWARGS = dict(
    fine_tune_layers=['reg_head_fc_2', 'reg_head_ln_2', 'reg_head_out']
)

OPT = Adam  # optimizer
LR = 5e-4  # optimizer learning rate
GRAD_CLIP_VAL = 5.0  # gradient clipping threshold

EARLY_STOP_PATIENCE = 7  # stop training after these many epochs without loss decrease
LRR_PATIENCE = 3  # multiply learning rate by LRR_FACTOR after these many epochs without loss decrease
LRR_FACTOR = 0.5
AUGMENTATIONS = [
    data_utils.EdgeFeaturesRandomRotation(always_apply=True)  # apply random in-plane rotation to all inputs
]

In [5]:
PATH_TO_FINETUNING_DATASET = (
    Path('example_data') /
    'finetuning' /
    'radial_1d_2d_inverted_9v'
)
TRAIN_GRAPHS_DIR_PATH = PATH_TO_FINETUNING_DATASET / 'train'
TEST_GRAPHS_DIR_PATH = PATH_TO_FINETUNING_DATASET / 'test'

### Load data

This example dataset contains graphs created from Surface Evolver .dmp files using `stressnet.se_output_to_graph` (see the "inference in silico" example notebook).

The same code can be used on graphs created from microscopy images using `stressnet.skeleton_to_graph` (see the "inference in vivo" example notebook).

In either case, graph data can be persisted using `stressnet.save_graph_data`.

**Randomly split the samples in the training directory into train and validation using an utility function**

In [6]:
train_dataset = pd.Series(list(TRAIN_GRAPHS_DIR_PATH.glob('*.npz')))
train_split = data_utils.split_data(
    train_dataset,
    val_split=VAL_SPLIT,
    test_split=0,  # we have evaluation samples in a different directory
    subsample=SUBSAMPLE,
    seed=SEED
)
train_split

DataSources(n_train=225,n_val=75,n_test=0)

**Fetch test samples**

In [7]:
test_dataset = pd.Series(list(TEST_GRAPHS_DIR_PATH.glob('*.npz')))
print(f'n_test={len(test_dataset)}')

n_test=100


**Instance data generators compatible with tensorflow-keras from the series of filepaths**

In [8]:
finetune_datagen = data_utils.GraphDataGenerator(
    train_split.train, shuffle=True, e_augmentations=AUGMENTATIONS
)
val_datagen = data_utils.GraphDataGenerator(
    train_split.val, shuffle=False, e_augmentations=None
)
test_datagen = data_utils.GraphDataGenerator(
    test_dataset, shuffle=False, e_augmentations=None
)

### Load StressNET with pretrained weights and evaluate the model on the test-set

In [9]:
# use fine-tuning params to make losses comparable (l2-reg differences)
base_model = stressnet.load_model(BASE_MODEL_WEIGHTS, **NET_KWARGS)
base_model.compile(loss=LOSS, metrics=METRICS)

In [10]:
base_loss, base_mae, base_mape = base_model.evaluate(x=test_datagen, steps=len(test_datagen), verbose=0)
del base_model # cleanup, we'll create a new instance for training
print(f'StressNET pretrained performance on the test set:')
print(f'  Loss: {base_loss:.4f} | MAE: {base_mae:.4f} | MAPE: {base_mape:.2f}% | support: {len(test_datagen)}')

StressNET pretrained performance on the test set:
  Loss: 0.0296 | MAE: 0.1040 | MAPE: 11.19% | support: 100


### Finetune StressNET on the training / validation splits

In [11]:
ft_model = stressnet.load_model(BASE_MODEL_WEIGHTS, **NET_KWARGS)

# most layers should be frozen
ft_model.summary()

Model: "StressNetV0.2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 edge_feats (InputLayer)        [(None, 32)]         0           []                               
                                                                                                  
 edge_feats_fc_1 (Dense)        (None, 512)          16384       ['edge_feats[0][0]']             
                                                                                                  
 edge_feats_ln_1 (LayerNormaliz  (None, 512)         1024        ['edge_feats_fc_1[0][0]']        
 ation)                                                                                           
                                                                                                  
 edge_feats_elu_1 (Activation)  (None, 512)          0           ['edge_feats_ln_1[0][

**Compile the model and configure training callbacks**

In [12]:
ft_model.compile(
    optimizer=OPT(learning_rate=LR, clipvalue=GRAD_CLIP_VAL),
    loss=LOSS,
    metrics=METRICS
)

learning_rate_reduction = ReduceLROnPlateau(
        monitor='val_loss',
        mode='min',
        min_delta=1e-4,
        patience=LRR_PATIENCE,
        verbose=1,
        factor=LRR_FACTOR,
        min_lr=1e-6,
    )

earlystop = EarlyStopping(
    monitor='val_loss',
    mode='min',
    min_delta=1e-4,
    patience=EARLY_STOP_PATIENCE,
    restore_best_weights=True,
    verbose=1,
)

callbacks = [earlystop, learning_rate_reduction, TerminateOnNaN()]

**Start the training loop**

In [13]:
history = ft_model.fit(
    x=finetune_datagen,
    validation_data=val_datagen,
    epochs=500, # we are using early-stopping, so just set this to a big number
    steps_per_epoch=len(finetune_datagen),
    validation_steps=len(val_datagen),
    callbacks=callbacks,
    verbose=1
)

Epoch 1/500
225/225 [==============================] - 17s 51ms/step - loss: 0.0050 - mean_absolute_error: 0.0728 - mean_absolute_percentage_error: 7.8637 - val_loss: 0.0041 - val_mean_absolute_error: 0.0414 - val_mean_absolute_percentage_error: 4.5075 - lr: 5.0000e-04
Epoch 2/500
225/225 [==============================] - 10s 44ms/step - loss: 0.0039 - mean_absolute_error: 0.0408 - mean_absolute_percentage_error: 4.4998 - val_loss: 0.0039 - val_mean_absolute_error: 0.0399 - val_mean_absolute_percentage_error: 4.3627 - lr: 5.0000e-04
Epoch 3/500
225/225 [==============================] - 11s 50ms/step - loss: 0.0036 - mean_absolute_error: 0.0382 - mean_absolute_percentage_error: 4.1742 - val_loss: 0.0036 - val_mean_absolute_error: 0.0400 - val_mean_absolute_percentage_error: 4.4288 - lr: 5.0000e-04
Epoch 4/500
225/225 [==============================] - 11s 50ms/step - loss: 0.0033 - mean_absolute_error: 0.0369 - mean_absolute_percentage_error: 3.9858 - val_loss: 0.0035 - val_mean_absol

### Evaluate the finetuned model on test split

In [14]:
ft_loss, ft_mae, ft_mape = ft_model.evaluate(x=test_datagen, steps=len(test_datagen), verbose=0)

print(f'StressNET pretrained performance on the test set:')
print(f'  Loss: {base_loss:.4f} | MAE: {base_mae:.4f} | MAPE: {base_mape:.2f}% | support: {len(test_datagen)}')

print(f'\nStressNET finetuned performance on the test set:')
print(f'  Loss: {ft_loss:.4f} | MAE: {ft_mae:.4f} | MAPE: {ft_mape:.2f}% | support: {len(test_datagen)}')

StressNET pretrained performance on the test set:
  Loss: 0.0296 | MAE: 0.1040 | MAPE: 11.19% | support: 100

StressNET finetuned performance on the test set:
  Loss: 0.0082 | MAE: 0.0506 | MAPE: 5.26% | support: 100


The model can be persisted with `ft_model.save(...)` (see `tensorflow.keras.Model` documentation)